# Three-dimensional nearly incompressible elasticity

GaLS and Taylor–Hood local spaces retain six rigid motions and impose the physical compressibility identity. The exact family keeps its force bounded as lambda tends to infinity.

The user-written local equations use P2 displacement and P2 Herrmann pressure,
with the full three-dimensional GaLS residual:

$$
\begin{aligned}
a_T((u,p),(v,q))={}&2\mu(\varepsilon(u),\varepsilon(v))_T
 -(p,\operatorname{div}v)_T-(q,\operatorname{div}u)_T\\
&-\lambda^{-1}(p,q)_T-\alpha_T h_T^2(R(u,p),R(v,q))_T,\\
R(u,p)={}&\operatorname{div}(2\mu\varepsilon(u))-\nabla p,\\
L_T(v,q)={}&(f,v)_T+\alpha_T h_T^2(f,R(v,q))_T,\\
B_T\lambda_T={}&\langle\lambda_T,v\rangle_{\partial T},\qquad
C_T(u,p)=-\langle u,\eta\rangle_{\partial T}.
\end{aligned}
$$

The existing pure inverse-bound kernel computes $\alpha_T$ at half its
sufficient physical tetrahedral bound, using six rigid modes; a two-dimensional
constant is not transferred here. Dyadic refinement two, P1 Bernstein traces,
quadrature degree six and the declared volume-centered rigid basis match the
consistency control. The global `Equation` supplies negative displacement
boundary moments and the physical pressure integral is $2|\Omega|$ at exact
incompressibility. The multiplier is negative Cauchy traction.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/2e0cc203c0d57e694fe22ec606cb01aa3a0dbb2f37ca273bc2ea09dbb3a983fc/55_gals3d-companion.zip"
COMPANION_SHA256 = "2e0cc203c0d57e694fe22ec606cb01aa3a0dbb2f37ca273bc2ea09dbb3a983fc"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("elasticity/55_gals3d.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.meshes.tetrahedron import TetraMesh

from functools import partial
from pymhm import Equation, MultiscaleProblem, assemble
from pymhm.fem.traces.triangle_3d import TriangularSkeleton
from examples.formulations.tetrahedral_vector import (
    TetrahedralVectorSpace, vector_boundary_moments,
    tetra_velocity_pressure_equations, tetra_velocity_fields,
    tetra_displacement_pressure_equations, tetra_displacement_fields,
)


In [ ]:
def displacement(x):
    """Affine solenoidal displacement with nonzero boundary data."""
    return np.column_stack((1+x[:,0], 2-x[:,1], x[:,0]+x[:,1]))

mesh = TetraMesh.unit_cube()
skeleton = TriangularSkeleton(mesh, degree=1)
local_space = TetrahedralVectorSpace(mesh, skeleton, 2, 2, 2, 6)
provider = partial(
    tetra_displacement_pressure_equations, data=local_space,
    source=(0.0, 0.0, 0.0), lame_lambda=np.inf,
)
boundary, fixed = vector_boundary_moments(skeleton, displacement, {}, 6)
problem = MultiscaleProblem(
    Equation(0, -np.r_[boundary, np.zeros(6 * len(mesh.cells))]),
    provider, range(len(mesh.cells)), 3 * skeleton.size, (6,) * len(mesh.cells), fixed=fixed,
)
with threadpool_limits(1):
    system = assemble(problem)
    gauge = system.mean_constraint(
        [record[2] for record in system.local_metadata], 2.0 * mesh.volumes.sum(),
    )
    coefficients = system.solve(constraints=(gauge,))
    patch = tetra_displacement_fields(system, coefficients, local_space)
    errors = {"displacement": patch.l2_error(displacement),
              "pressure": patch.pressure_l2_error(2.0),
              "compressibility": patch.compressibility_l2()}
assert max(errors.values()) < 1e-10
errors


Pressure means are prescribed only at exact incompressibility on a full displacement boundary. Finite lambda fixes the weighted pressure integral through the displacement boundary flux. Cauchy traction is symmetric; it differs from the grad-grad flow pseudotraction.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    folder = ROOT / "examples/results/gals3d"
    report = json.loads((folder / "campaign.json").read_text())
    rows = report["rows"]
    assert len(rows) == 36
    for row in rows:
        assert hashlib.sha256((folder / row["fields"]).read_bytes()).hexdigest() == row["fields_sha256"]
        assert row["max_error_quadrature_relative_difference"] < 1e-8
        assert row["backward_residual"] < 1e-8
    for case in ["gals-p1", "gals-p2", "th-p2"]:
        refined = [row for row in rows if row["case"] == case and row["study"] == "refinement"]
        assert len(refined) == 5
        assert refined[-1]["displacement_l2"] < refined[0]["displacement_l2"]
        assert refined[-1]["pressure_l2"] < refined[0]["pressure_l2"]
    [(row["case"], row["lame_lambda"], row["displacement_l2"], row["pressure_l2"])
     for row in rows if row["study"] == "lambda"]

    primal = json.loads((ROOT / "examples/results/gals3d/primal/campaign.json").read_text())
    assert len(primal["primal_control"]) == 4
    assert all(row["local_refinement_precision"] == "extended" for row in primal["primal_control"])


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    figures = ROOT / "docs/figures/gals3d"
    for name in ["refinement", "lambda", "gals-p1-fields", "gals-p2-fields", "th-p2-fields"]:
        display(Image(filename=str(figures / f"{name}.png")))


The refinement and parameter studies are manufactured 3D validation, not reproductions of the paper's two-dimensional figures. The primal comparison uses the same operator, material and boundary data, with finite lambda. These finite tests do not establish a uniform inf-sup bound for every mesh. Raw stresses are not an H(div) reconstruction.